# Compare curve vs. block representation: stats only

Assumes every experiment already has **both** a curve-representation and a block-representation trained ensemble under `MODEL_ROOT`, each with a `stats.pkl` saved alongside it (i.e. `cnnpz.get_all_stats(..., save=True, saveroot=save_dir + "/stats.pkl")` was run once for each). This notebook does no data loading, no filter-bank/block reconstruction, and no model inference -- it just reads the saved `(stats, redshift_stats, imag_stats)` pickles with `cnnpz.read_stats` and compares them.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cnnpz

cnnpz.set_plot_style()

In [ ]:
# Parametric paths, same convention as CNN_photoz-ensemble-cardinal.ipynb
USER = os.environ.get("USER", "jaimerz")
PSCRATCH = os.environ.get("PSCRATCH", f"/pscratch/sd/{USER[0]}/{USER}")

MODEL_ROOT = os.path.join(PSCRATCH, "cnnpz", "noisy_Cardinal", "models")
# pre-training/fine-tuning models were saved under a different root in the source notebook
PRETRAIN_MODEL_ROOT = "/pscratch/sd/q/qhang/cnnpz/noisy_Cardinal/models"

# (experiment name, root, curve subdir, block subdir)
experiments = [
    ("Y1 complete", MODEL_ROOT, "y1_complete_curve_ensemble_CNN_6layers", "y1_complete_ensemble_CNN_6layers"),
    ("Y1 NIR-dropout", MODEL_ROOT, "y1_misnir_curve_ensemble_CNN_6layers", "y1_misnir_ensemble_CNN_6layers"),
    ("Y10 complete", MODEL_ROOT, "y10_complete_curve_ensemble_CNN_6layers", "y10_complete_ensemble_CNN_6layers"),
    ("Y10 NIR-dropout", MODEL_ROOT, "y10_misnir_curve_ensemble_CNN_6layers", "y10_misnir_ensemble_CNN_6layers"),
    ("Y1 spec-select", MODEL_ROOT, "y1_specsel_curve_ensemble_CNN_6layers", "y1_specsel_ensemble_CNN_6layers"),
    ("Y10 spec-select", MODEL_ROOT, "y10_specsel_curve_ensemble_CNN_6layers", "y10_specsel_ensemble_CNN_6layers"),
    ("Y10 pre-training", PRETRAIN_MODEL_ROOT, "y10_pretrain_curve_ensemble_CNN_6layers", "y10_pretrain_ensemble_CNN_6layers"),
    ("Y10 fine-tune", PRETRAIN_MODEL_ROOT, "y10_finetune_curve_ensemble_CNN_6layers", "y10_finetune_ensemble_CNN_6layers"),
]

In [ ]:
# same binning used everywhere else in the project when these stats were generated
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)

results = []

for name, root, curve_subdir, block_subdir in experiments:
    curve_stats, curve_z_stats, curve_i_stats = cnnpz.read_stats(os.path.join(root, curve_subdir, "stats.pkl"))
    block_stats, block_z_stats, block_i_stats = cnnpz.read_stats(os.path.join(root, block_subdir, "stats.pkl"))

    print(f"=== {name} ===")
    print(cnnpz.stats_to_markdown(block_stats, curve_stats, data_title=("block", "curve")))
    cnnpz.compare_binned_stats(redshift_bins, imag_bins, block_z_stats, block_i_stats, curve_z_stats, curve_i_stats)
    plt.suptitle(name)
    plt.show()

    # stats tuple is (mean, mean_err, std, outlier_rate, abs_outlier_rate) -- std is biweight sigma_z
    sigma_block, sigma_curve = block_stats[2], curve_stats[2]
    results.append({
        "experiment": name,
        "sigma_z_block": sigma_block,
        "sigma_z_curve": sigma_curve,
        "outlier_rate_block": block_stats[3],
        "outlier_rate_curve": curve_stats[3],
        "winner (lower sigma_z)": "curve" if sigma_curve < sigma_block else "block",
    })

In [ ]:
summary = pd.DataFrame(results)
n_curve_wins = (summary["winner (lower sigma_z)"] == "curve").sum()
print(summary.to_string(index=False))
print(f"\ncurve representation wins {n_curve_wins}/{len(summary)} experiments (lower biweight sigma_z)")
summary